# Delhi AQI World Model — Technical Walkthrough

This notebook rebuilds the current project one layer at a time. Run each cell with **Shift+Enter**. The goal is to make every assumption, data transformation, and visual output inspectable.

> The current PM2.5 dynamics are synthetic. Real roads and monitoring-station locations provide geographic context, but the concentration model is not yet calibrated for forecasting.

## 1. How the pieces fit together

- `src/model.py` defines the grid, simulation assumptions, emissions, and dispersion.
- `src/geography.py` loads real OSM roads and OpenAQ monitoring sites.
- `src/utils.py` turns simulated concentration arrays into maps.
- `src/server.py` is the command-line entry point.
- `tests/` checks geometry, output shape, valid ranges, and reproducibility.

The important separation is **geography versus dynamics**: real map features describe where the model runs; the simulation describes how PM2.5 changes through time.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.geography import (
    bounds_from_config,
    group_station_sites,
    load_pm25_stations,
    load_road_segments,
    plot_context_map,
)
from src.model import SimulationConfig, simulate_pm25
from src.utils import plot_hour

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)
print("Walkthrough imports loaded successfully.")

## 2. Define one experiment

`SimulationConfig` holds assumptions instead of scattering constants through the code. Our first experiment covers a 5 km square centered on Connaught Place, split into 100 cells, with one state update per hour.

In [ ]:
config = SimulationConfig()
config, config.cell_size_km, config.cell_count

## 3. Convert kilometres into geographic bounds

The model grid uses kilometres, while OpenStreetMap and OpenAQ use latitude and longitude. `bounds_from_config` approximates how many degrees span 2.5 km north/south and east/west at Delhi's latitude. Longitude needs a cosine correction because meridians get closer together away from the equator.

In [ ]:
bounds = bounds_from_config(config)
bounds

## 4. Load real roads and monitoring locations

The road snapshot contains OSM ways downloaded through the Overpass API. The station CSV came from OpenAQ v3. We keep only stations inside our square that report PM2.5, then group duplicate provider records representing the same named site.

In [ ]:
roads_path = PROJECT_ROOT / "data/connaught_place_roads.json"
stations_path = PROJECT_ROOT / "data/delhi_stations.csv"

roads = load_road_segments(roads_path)
station_records = load_pm25_stations(stations_path, bounds)
station_sites = group_station_sites(station_records)

print(f"Road segments: {len(roads):,}")
print(f"PM2.5 station records: {len(station_records)}")
print(f"Grouped physical sites: {len(station_sites)}")
station_sites

In [ ]:
context_figure = plot_context_map(roads, station_records, config)
context_figure

## 5. Run the synthetic PM2.5 dynamics

For every hour, `simulate_pm25` performs four conceptual operations:

1. **Diffuse:** blend each cell with its four neighbours.
2. **Advect:** shift some concentration north-east to imitate wind.
3. **Emit:** add synthetic road and hotspot emissions, with morning/evening traffic peaks.
4. **Remove and perturb:** decay pollution above the background level and add seeded weather noise.

The returned NumPy array is indexed as `[hour, row, column]`.

In [ ]:
history = simulate_pm25(config)
print("Array shape:", history.shape)
print(f"Minimum: {history.min():.1f} µg/m³")
print(f"Maximum: {history.max():.1f} µg/m³")
print(f"Overall mean: {history.mean():.1f} µg/m³")

## 6. Inspect how the grid changes through the day

A shared color scale is essential: without it, each panel would stretch its own colors and make weak and strong pollution periods look deceptively similar.

In [ ]:
hours_to_show = [0, 6, 12, 19]
figure, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
vmin, vmax = float(history.min()), float(history.max())

for axis, hour in zip(axes, hours_to_show):
    image = axis.imshow(
        history[hour], origin="lower", cmap="magma_r",
        vmin=vmin, vmax=vmax, extent=(0, 5, 0, 5),
    )
    axis.set_title(f"Hour {hour:02d}:00")
    axis.set_xlabel("km east")

axes[0].set_ylabel("km north")
figure.colorbar(image, ax=axes, label="PM2.5 (µg/m³)", shrink=0.8)
figure

## 7. Reduce the grid to interpretable hourly metrics

Spatial maps show *where* pollution accumulates. Aggregated mean and maximum values show *when* the model becomes most polluted and expose the synthetic rush-hour pattern.

In [ ]:
hourly_mean = history.mean(axis=(1, 2))
hourly_max = history.max(axis=(1, 2))

figure, axis = plt.subplots(figsize=(10, 4))
axis.plot(hourly_mean, marker="o", label="Grid mean")
axis.plot(hourly_max, marker="o", label="Grid maximum")
axis.set(xlabel="Simulation hour", ylabel="PM2.5 (µg/m³)", title="Synthetic daily PM2.5 profile")
axis.set_xticks(range(0, config.hours, 2))
axis.grid(alpha=0.25)
axis.legend()
figure

## 8. What is real, synthetic, and next

**Real now:** Connaught Place location, study boundary, OSM roads, OpenAQ station locations, and PM2.5 capability metadata.

**Synthetic now:** starting concentrations, emissions, traffic schedule, wind direction, diffusion, removal, and weather noise.

**Next milestone:** retrieve timestamped station measurements, assign each monitor to a grid cell, and interpolate those observations to initialize the full grid. After that, road density can replace the hand-drawn synthetic emission corridors.